# The SSL criterion as a kernel-selection tool (VAMP-2 / spectral contrastive)

**Status before this notebook:** the spectral contrastive loss was only ever the encoders' *training*
objective; VAMP was never computed. Neither has been used for *selection* anywhere — selection has always
been the Kostic-metric composite. This notebook closes that gap per the supervisor prompt: use the ICLR-2026
SSL criterion for kernel selection in the NeurIPS-2023 setting.

**Why it applies to fixed kernels too.** For any feature map φ (including the implicit RKHS one), the
spectral contrastive loss on lagged pairs is
L_SC = (1/(N(N−1))) Σ_{i≠j} ⟨φ(xᵢ), φ(yⱼ)⟩² − (2/N) Σᵢ ⟨φ(xᵢ), φ(yᵢ)⟩,
and ⟨φ(x), φ(y)⟩ = k(x, y) — so L_SC is computable from the cross-kernel matrix alone. Its population
minimiser captures the operator's top **singular** subspace, and the matching score is
**VAMP-2(r) = Σᵢ₌₁..ᵣ σᵢ²** of the whitened operator C_X^{−1/2} C_XY C_Y^{−1/2} (canonical correlations),
estimable from Gram matrices. Lower SC = better; higher VAMP-2 = better.

**The study question** (and the link to the thesis narrative): both criteria measure *singular-value energy*
captured — not eigen-spectral fidelity. For normal operators the two coincide; for non-normal ones they can
diverge. Part 2 tests, for **every** system × kernel family sweep and the SSL encoders, whether
energy-based selection agrees with (a) true eigenvalue error on the analytic systems, (b) the Kostic metrics,
and (c) the unified score.

**Part 1 needs your kooplearn environment** (data generators only — no model fitting; ≈ pseudospectra-rerun
cost). Part 2 is CSV-only and runs on Part 1's output. Outputs → `analysis/vamp_selection/`.

In [ ]:
import os
import re

import numpy as np
import pandas as pd
from scipy.linalg import expm
from scipy.spatial.distance import cdist
from scipy.stats import spearmanr

BASE = os.path.join("..", "benchmarks")
AN = os.path.join("..", "analysis")
OUT = os.path.join(AN, "vamp_selection")
os.makedirs(OUT, exist_ok=True)

N_TRIALS = 3  # fresh data draws per criterion estimate
N_TRAIN, N_VAL = 2000, 500
N_VAMP = 400  # subsample for the whitened SVD (n^3 cost)
R_VAMP = 3  # rank, matching the analysed modes 1-3
EPS_WHITEN = 1e-6
HERM_R, HERM_M = 3, 10  # match your hermite notebook's (r, M) if different


def norm_method(s):
    s = str(s).strip()
    return "PCR" if s.startswith("Principal Components") else s


## Kernel matrices (vectorised; Hermite construction copied verbatim from your notebook)

In [ ]:
def rbf_K(X, Y, gamma):
    return np.exp(-gamma * cdist(np.atleast_2d(X), np.atleast_2d(Y), "sqeuclidean"))


def linear_K(X, Y):
    return np.atleast_2d(X) @ np.atleast_2d(Y).T


def poly_K(X, Y, p, gamma=None, coef0=1.0):
    # kooplearn KernelRidge(kernel="poly", degree=p) delegates to the sklearn
    # convention (gamma = 1/n_features, coef0 = 1); adjust here if yours differs
    X, Y = np.atleast_2d(X), np.atleast_2d(Y)
    g = gamma if gamma is not None else 1.0 / X.shape[1]
    return (g * (X @ Y.T) + coef0) ** p


# --- Hermite kernel, verbatim helpers from kernel_analysis_hermite.ipynb ------
def hermite_features(x, M):
    x = np.asarray(x).reshape(-1)
    f = np.empty((x.shape[0], M))
    f[:, 0] = 1.0
    if M > 1:
        f[:, 1] = x
    for n in range(2, M):
        f[:, n] = (x * f[:, n - 1] - np.sqrt(n - 1) * f[:, n - 2]) / np.sqrt(n)
    return f


def kernel_permutation(kind, r, M):
    pi = np.arange(M)
    if kind == "good":
        return pi
    if kind == "bad":
        pi2 = pi.copy()
        a, b = np.arange(r), np.arange(r, 2 * r)
        pi2[a], pi2[b] = b[::-1], a[::-1]
        return pi2
    if kind == "ugly":
        return pi[::-1]
    raise ValueError(kind)


def make_weights(kind, r, M):
    mu = np.exp(-np.arange(M))
    nu = 1.0 if kind == "good" else (1.0 / (r**2) if kind == "bad" else r**2)
    pi = kernel_permutation(kind, r, M)
    return mu[pi] ** (2.0 * nu)


def hermite_K(X, Y, kind, r=HERM_R, M=HERM_M):
    # NOTE: matches your notebook's behaviour — features on the FIRST coordinate
    # (hermite_features(x)[0] in the original per-pair kernel)
    x1 = np.atleast_2d(X)[:, 0]
    y1 = np.atleast_2d(Y)[:, 0]
    w = make_weights(kind, r, M)
    return (hermite_features(x1, M) * w) @ hermite_features(y1, M).T


# --- graph kernels (from kernel_analysis_graphs.ipynb) ------------------------
def graph_laplacian_from_P(P):
    A = 0.5 * (P + P.T)
    A = (A > 0).astype(float)
    np.fill_diagonal(A, 0.0)
    D = np.diag(A.sum(axis=1))
    return D - A


def graph_K_matrix(P, kernel_name):
    L = graph_laplacian_from_P(P)
    if kernel_name == "delta":
        return np.eye(P.shape[0])
    m = re.match(r"diffusion\(tau=([0-9.eE+-]+)\)", kernel_name)
    if m:
        return expm(-float(m.group(1)) * L)
    m = re.match(r"resolvent\(beta=([0-9.eE+-]+)\)", kernel_name)
    if m:
        return np.linalg.inv(L + float(m.group(1)) * np.eye(L.shape[0]))
    raise ValueError(kernel_name)


## The two criteria

In [ ]:
def sc_criterion(K_xy_val):
    """Spectral contrastive loss from the cross-kernel matrix on lagged
    validation pairs (lower = better)."""
    K = np.asarray(K_xy_val, float)
    N = K.shape[0]
    diag = np.diag(K)
    off_sq = (K**2).sum() - (diag**2).sum()
    return float(off_sq / (N * (N - 1)) - 2.0 * diag.mean())


def vamp2_r(K_X, K_Y, K_XY, r=R_VAMP, eps=EPS_WHITEN):
    """VAMP-2(r) = sum of top-r squared singular values of the whitened
    operator, estimated from Gram matrices (higher = better)."""
    n = K_X.shape[0]

    def inv_sqrt(K):
        w, V = np.linalg.eigh(K / n)
        w = np.clip(w, eps, None)
        return (V / np.sqrt(w)) @ V.T / np.sqrt(n)

    A = inv_sqrt(K_X) @ (K_XY) @ inv_sqrt(K_Y)
    s = np.linalg.svd(A, compute_uv=False)
    s = np.clip(s, 0, 1.0 + 1e-6)  # canonical correlations
    return float(np.sum(s[:r] ** 2))


def criteria_for(K_fn, X_tr, Y_tr, X_val, Y_val):
    """Both criteria for one kernel on one data draw."""
    sub = slice(0, N_VAMP)
    Kx = K_fn(X_tr[sub], X_tr[sub])
    Ky = K_fn(Y_tr[sub], Y_tr[sub])
    Kxy = K_fn(X_tr[sub], Y_tr[sub])
    return {"sc": sc_criterion(K_fn(X_val, Y_val)), "vamp2": vamp2_r(Kx, Ky, Kxy)}


## Data generators (identical to the rerun notebook / originals — same seeds)

In [ ]:
from kooplearn.datasets import (
    make_duffing,
    make_linear_system,
    make_logistic_map,
    make_prinz_potential,
)


def pairs(arr, n_train=N_TRAIN, n_val=N_VAL):
    tr = arr[:n_train]
    va = arr[n_train : n_train + n_val]
    return tr[:-1], tr[1:], va[:-1], va[1:]


def gen_ou(kind, trial):
    a, b = np.exp(-1.0 * 1e-4), np.sqrt(1 - np.exp(-2e-4))
    rng = np.random.default_rng(10_000 + trial)
    n = (N_TRAIN + N_VAL) * 100
    x = np.empty(n)
    x[0] = 0.0
    for t in range(n - 1):
        x[t + 1] = a * x[t] + b * rng.standard_normal()
    return pairs(x[::100][:, None])


def gen_langevin(kind, trial):
    n = (N_TRAIN + N_VAL + 200) * 100
    d = make_prinz_potential(X0=0, n_steps=n, gamma=1.0, sigma=2.0, random_state=10_000 + trial)
    arr = d.values[::100][200:]
    if arr.ndim == 1:
        arr = arr[:, None]
    return pairs(arr)


def gen_rot(kind, trial):
    omega = float(kind.split("=")[1])
    rng = np.random.default_rng(10_000 + 100 * int(10 * omega) + trial)
    J = np.array([[0.0, omega], [-omega, 0.0]])
    n = (N_TRAIN + N_VAL) * 100 + 200
    x = np.zeros((n, 2))
    ns = np.sqrt(2e-4)
    for t in range(n - 1):
        g = np.array([x[t, 0] * (x[t, 0] ** 2 - 1), x[t, 1] * (x[t, 1] ** 2 - 1)])
        x[t + 1] = x[t] + (-g + J @ x[t]) * 1e-4 + ns * rng.standard_normal(2)
    return pairs(x[200:][::100])


def gen_duffing(kind, trial):
    omega = float(kind.split("=")[1])
    rng = np.random.default_rng(10_000 + trial)
    X0 = rng.uniform(-0.5, 0.5, 2)
    d = make_duffing(
        X0=X0,
        n_steps=(N_TRAIN + N_VAL) * 20,
        dt=1e-3,
        alpha=-1.0,
        beta=1.0,
        gamma=0.3,
        delta=0.2,
        omega=omega,
    )
    return pairs(d.values[::20])


def gen_logistic(kind, trial):
    r = float(kind.split("=")[1])
    rng = np.random.default_rng(10_000 + trial)
    d = make_logistic_map(
        X0=float(rng.uniform(0.05, 0.95)),
        n_steps=(N_TRAIN + N_VAL) * 20,
        r=r,
        M=20,
        dt=1,
        random_state=10_000 + trial,
    )
    arr = d.values[::20]
    if arr.ndim == 1:
        arr = arr[:, None]
    return pairs(arr)


A_CASES = {"A=rev": np.diag([-1.0, -2.0]), "A=rot": np.array([[-1.0, 2.0], [-2.0, -1.5]])}


def gen_linear(kind, trial):
    A = A_CASES[kind]
    A_disc = np.eye(2) + 1e-3 * A
    d = make_linear_system(
        X0=np.zeros(2),
        A=A_disc,
        n_steps=(N_TRAIN + N_VAL) * 20,
        noise=np.sqrt(2e-3),
        random_state=1000 + trial,
    )
    return pairs(d.values[::20])


def harmonic_oscillator_flow(q0, p0, t, omega=1.0, damping=0.0):
    t = np.asarray(t, float)
    if damping == 0.0:
        return (
            q0 * np.cos(omega * t) + (p0 / omega) * np.sin(omega * t),
            -q0 * omega * np.sin(omega * t) + p0 * np.cos(omega * t),
        )
    disc = omega**2 - damping**2
    if disc > 0:
        om = np.sqrt(disc)
        A_, B_ = q0, (p0 + damping * q0) / om
        e = np.exp(-damping * t)
        q = e * (A_ * np.cos(om * t) + B_ * np.sin(om * t))
        dq = -A_ * om * np.sin(om * t) + B_ * om * np.cos(om * t)
        return q, e * (dq - damping * (A_ * np.cos(om * t) + B_ * np.sin(om * t)))
    if disc == 0:
        A_, B_ = q0, p0 + damping * q0
        e = np.exp(-damping * t)
        return e * (A_ + B_ * t), e * (B_ - damping * (A_ + B_ * t))
    om = np.sqrt(-disc)
    r1, r2 = -damping + om, -damping - om
    C1 = (p0 - r2 * q0) / (r1 - r2)
    C2 = q0 - C1
    return (
        C1 * np.exp(r1 * t) + C2 * np.exp(r2 * t),
        C1 * r1 * np.exp(r1 * t) + C2 * r2 * np.exp(r2 * t),
    )


def gen_ho(kind, trial):
    # 256 disk-sampled initial conditions, analytic flow, lag pairs pooled
    # across trajectories (aligned with generate_ho in your HO cell)
    damping = float(kind.split("=")[1])
    rng = np.random.default_rng(10_000 + trial)
    n_init = 256
    t_max = (10 * 2 * np.pi) if damping == 0.0 else 4.0 / damping
    steps = (N_TRAIN + N_VAL) // n_init + 2
    t_grid = np.linspace(0.0, t_max, steps)
    ang = rng.uniform(0, 2 * np.pi, n_init)
    rad = np.sqrt(rng.uniform(0, 1, n_init))
    Xs, Ys = [], []
    for q0, p0 in zip(rad * np.cos(ang), rad * np.sin(ang)):
        q, p = harmonic_oscillator_flow(q0, p0, t_grid, 1.0, damping)
        tr = np.stack([q, p], -1)
        Xs.append(tr[:-1])
        Ys.append(tr[1:])
    X, Y = np.concatenate(Xs), np.concatenate(Ys)
    ntr = min(N_TRAIN, len(X) - N_VAL)
    return X[:ntr], Y[:ntr], X[ntr : ntr + N_VAL], Y[ntr : ntr + N_VAL]


def cycle_P(n=12):
    P = np.zeros((n, n))
    for i in range(n):
        P[i, (i - 1) % n] = P[i, (i + 1) % n] = 0.5
    return P


def path_P(n=12):
    P = np.zeros((n, n))
    for i in range(n):
        nbrs = [j for j in (i - 1, i + 1) if 0 <= j < n]
        for j in nbrs:
            P[i, j] = 1.0 / len(nbrs)
    return P


GRAPHS = {"cycle_12": cycle_P(), "path_12": path_P()}


def gen_graph(kind, trial):
    P = GRAPHS[kind]
    rng = np.random.default_rng(30_000 + trial)
    s = np.empty(N_TRAIN + N_VAL, int)
    s[0] = 0
    for t in range(len(s) - 1):
        s[t + 1] = rng.choice(P.shape[0], p=P[s[t]])
    return pairs(s[:, None])


SYSTEMS = {
    "ou": (gen_ou, ["all"]),
    "langevin": (gen_langevin, ["all"]),
    "rot": (gen_rot, [f"omega={w}" for w in (0.0, 0.5, 1.0, 2.0)]),
    "duffing": (gen_duffing, [f"omega={w}" for w in (0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0)]),
    "logistic": (gen_logistic, [f"r={r}" for r in (0.0, 0.5, 1.2, 2.5, 3.2, 4.0)]),
    "linear": (gen_linear, ["A=rev", "A=rot"]),
    "ho": (gen_ho, [f"damping={d}" for d in (0.0, 0.1, 0.3, 0.5, 1.0, 1.5)]),
    "graph": (gen_graph, ["cycle_12", "path_12"]),
}


## Candidate kernels — parsed from your own benchmark CSVs (exact same sweeps)

In [ ]:
SCORE_FILES = {
    ("rbf", "ou"): "rbf/ou/ou_rbf_scores.csv",
    ("rbf", "langevin"): "rbf/langevin/langevin_rbf_scores.csv",
    ("rbf", "rot"): "rbf/rot/rot_rbf_scores.csv",
    ("rbf", "duffing"): "rbf/duffing/duf_rbf_scores.csv",
    ("rbf", "logistic"): "rbf/logistic/log_rbf_scores.csv",
    ("rbf", "linear"): "rbf/linear/lin_rbf_scores.csv",
    ("rbf", "ho"): "rbf/ho/ho-rbf_scores.csv",
    ("graph_delta", "graph"): "graph/delta/graph_delta_scores.csv",
    ("graph_diffusion", "graph"): "graph/diffusion/graph_diff_scores.csv",
    ("graph_resolvent", "graph"): "graph/resolvent/graph_res_scores.csv",
}


def kernel_fn(family, name):
    """Return a vectorised K(X, Y) for a benchmark kernel name."""
    if family == "rbf":
        g = float(re.search(r"gamma=([0-9.eE+-]+)", name).group(1))
        return lambda X, Y: rbf_K(X, Y, g)
    if family == "linear":
        return linear_K
    if family == "poly":
        p = int(re.search(r"p=(\d+)", name).group(1))
        return lambda X, Y: poly_K(X, Y, p)
    if family == "hermite":
        kind = re.search(r"hermite\((\w+)\)", name).group(1)
        return lambda X, Y: hermite_K(X, Y, kind)
    raise ValueError(family)


def candidates_for(system):
    """(family, kernel_name) candidates, matching the original sweeps."""
    out = []
    if system != "graph":
        rel = SCORE_FILES.get(("rbf", system))
        if rel and os.path.exists(os.path.join(BASE, rel)):
            for name in pd.read_csv(os.path.join(BASE, rel))["kernel"].unique():
                out.append(("rbf", str(name)))
        out.append(("linear", "linear"))
        out += [("poly", f"poly(p={p})") for p in (1, 2, 3, 4)]
        out += [("hermite", f"hermite({k})") for k in ("good", "bad", "ugly")]
    else:
        for fam in ("graph_delta", "graph_diffusion", "graph_resolvent"):
            rel = SCORE_FILES[(fam, "graph")]
            for name in pd.read_csv(os.path.join(BASE, rel))["kernel"].unique():
                out.append((fam, str(name)))
    return out


## Part 1 — compute both criteria for every system × kernel × trial *(needs kooplearn env)*

In [ ]:
records = []
for system, (gen, kinds) in SYSTEMS.items():
    cands = candidates_for(system)
    for kind in kinds:
        for trial in range(N_TRIALS):
            X_tr, Y_tr, X_val, Y_val = gen(kind, trial)
            for family, name in cands:
                if system == "graph":
                    Kmat = graph_K_matrix(GRAPHS[kind], name)
                    kf = lambda A, B, K=Kmat: K[A[:, 0].astype(int)][:, B[:, 0].astype(int)]
                else:
                    kf = kernel_fn(family, name)
                try:
                    c = criteria_for(kf, X_tr, Y_tr, X_val, Y_val)
                except Exception as e:
                    print(f"  skip {system}/{kind}/{name}: {e}")
                    continue
                records.append(
                    {
                        "system": system,
                        "kind": kind,
                        "family": family,
                        "kernel": name,
                        "trial": trial,
                    }
                    | c
                )
        print(f"[{system}] {kind}: {len(cands)} kernels x trial done")

crit = pd.DataFrame(records)
crit_agg = crit.groupby(["system", "kind", "family", "kernel"], as_index=False)[
    ["sc", "vamp2"]
].mean()
crit.to_csv(os.path.join(OUT, "vamp_sc_criteria_trials.csv"), index=False)
crit_agg.to_csv(os.path.join(OUT, "vamp_sc_criteria.csv"), index=False)
print(f"\n{len(crit_agg)} (system, kind, kernel) criteria computed")


[ou] all: 58 kernels x trial done
[langevin] all: 58 kernels x trial done
[rot] omega=0.0: 208 kernels x trial done
[rot] omega=0.5: 208 kernels x trial done
[rot] omega=1.0: 208 kernels x trial done
[rot] omega=2.0: 208 kernels x trial done
[duffing] omega=0.5: 358 kernels x trial done
[duffing] omega=0.8: 358 kernels x trial done
[duffing] omega=1.0: 358 kernels x trial done
[duffing] omega=1.2: 358 kernels x trial done
[duffing] omega=1.5: 358 kernels x trial done
[duffing] omega=2.0: 358 kernels x trial done
[duffing] omega=3.0: 358 kernels x trial done
[logistic] r=0.0: 15 kernels x trial done
[logistic] r=0.5: 15 kernels x trial done
[logistic] r=1.2: 15 kernels x trial done
[logistic] r=2.5: 15 kernels x trial done
[logistic] r=3.2: 15 kernels x trial done
[logistic] r=4.0: 15 kernels x trial done
[linear] A=rev: 108 kernels x trial done
[linear] A=rot: 108 kernels x trial done
[ho] damping=0.0: 308 kernels x trial done
[ho] damping=0.1: 308 kernels x trial done
[ho] damping=0.3

## SSL encoders — SC criterion from saved training diagnostics *(CSV-only)*

The encoders' spectral-contrastive **validation loss is already saved** per configuration — it IS the
criterion, no recomputation needed. (Optional: VAMP-2 for encoders needs latents; if wanted, add
`vamp2_r` on `Phi_X/Phi_Y` inside `analyse_encoder` of `ssl_analysis_full.ipynb` and re-run that sweep —
one line, marked there as the natural extension.)

In [ ]:
ssl_frames = []
for s in ["langevin", "duffing", "linear", "graph"]:
    p = os.path.join(AN, "ssl", f"{s}_ssl_training_diagnostics.csv")
    if os.path.exists(p):
        d = pd.read_csv(p)
        d["system"] = s
        ssl_frames.append(d)
ssl_sc = pd.concat(ssl_frames, ignore_index=True)
ssl_sc = (
    ssl_sc.groupby(["system", "kind", "kernel"], as_index=False)["final_val_loss"]
    .mean()
    .rename(columns={"final_val_loss": "sc"})
)
ssl_sc["family"] = "ssl_encoder"
ssl_sc.to_csv(os.path.join(OUT, "ssl_sc_criteria.csv"), index=False)
print(f"SSL SC criteria for {len(ssl_sc)} encoder configurations")


SSL SC criteria for 24 encoder configurations


## Part 2 — the selection study *(CSV-only; reconsiders ALL results)*

For every (system, kind) pool: rank candidates by SC (lower better) and VAMP-2 (higher better), then test the
ranking against (a) **true eigenvalue error** where analytic, (b) the Kostic metrics (bias, distortion,
spurious), (c) the unified score. Same `selection_quality` conventions as everywhere else: Spearman, top-1
regret with 10% tolerance.

In [10]:
def norm_kind(k):
    try:
        float(str(k))
        return "all"
    except ValueError:
        return str(k)


crit_agg = pd.read_csv(os.path.join(OUT, "vamp_sc_criteria.csv"))

# ground truths / reference metrics
uni = pd.read_csv(os.path.join(AN, "unified_scoring_database.csv"))
uni["kind"] = uni["kind"].map(norm_kind)
uni["method"] = uni["method"].map(norm_method)
uni_m = uni.groupby(["family", "system", "kind", "kernel"], as_index=False)[
    ["agg_bias_mean", "agg_dist_mean", "mean_spurious_residual_count", "score_unified"]
].min()

te_frames = [pd.read_csv(os.path.join(AN, "true_spectrum_validation.csv"))]
lgv = pd.read_csv(os.path.join(AN, "reruns", "langevin_true_spectrum_validation.csv"))
lgv["system"], lgv["kind"] = "langevin", "all"
te_frames.append(lgv)
te = pd.concat(te_frames, ignore_index=True)
te["kind"] = te["kind"].map(norm_kind)
te = te.groupby(["family", "system", "kind", "kernel"], as_index=False)["true_eig_err"].mean()

crit_agg["kind_n"] = crit_agg["kind"].map(norm_kind)
J = crit_agg.merge(
    uni_m,
    left_on=["family", "system", "kind_n", "kernel"],
    right_on=["family", "system", "kind", "kernel"],
    how="left",
    suffixes=("", "_u"),
).merge(
    te,
    left_on=["family", "system", "kind_n", "kernel"],
    right_on=["family", "system", "kind", "kernel"],
    how="left",
    suffixes=("", "_t"),
)
J.to_csv(os.path.join(OUT, "criteria_joined.csv"), index=False)
print(
    f"joined: {len(J)} rows | with true_eig_err: {J['true_eig_err'].notna().sum()} "
    f"| with unified metrics: {J['agg_bias_mean'].notna().sum()}"
)


def sel_quality(g, crit_col, truth_col, higher_better):
    g = g.dropna(subset=[crit_col, truth_col])
    if len(g) < 3 or g[truth_col].nunique() < 2:
        return None
    sgn = -1.0 if higher_better else 1.0
    pick = g.loc[(sgn * g[crit_col]).idxmin()]
    best = g[truth_col].min()
    return {
        "n": len(g),
        "spearman": spearmanr(sgn * g[crit_col], g[truth_col]).statistic,
        "top1_ok": (pick[truth_col] - best < 1e-9)
        or (best > 0 and (pick[truth_col] - best) / best < 0.10),
        "rel_regret": (pick[truth_col] - best) / best if best > 0 else np.nan,
    }


rows = []
for (sys_, kind), g in J.groupby(["system", "kind_n"]):
    for crit_col, hb in [("sc", False), ("vamp2", True)]:
        for truth, label in [
            ("true_eig_err", "true eigenvalue error"),
            ("agg_bias_mean", "spectral bias (s-hat)"),
            ("mean_spurious_residual_count", "spurious count"),
            ("score_unified", "unified score"),
        ]:
            q = sel_quality(g, crit_col, truth, hb)
            if q:
                rows.append(
                    {"system": sys_, "kind": kind, "criterion": crit_col, "truth": label} | q
                )
ev = pd.DataFrame(rows)
ev.to_csv(os.path.join(OUT, "criterion_selection_quality.csv"), index=False)

print("\n=== Does the SSL criterion select good kernels? (median over pools) ===")
summ = (
    ev.groupby(["criterion", "truth"])
    .agg(
        median_spearman=("spearman", "median"),
        top1_ok=("top1_ok", "mean"),
        median_rel_regret=("rel_regret", "median"),
        n_pools=("n", "size"),
    )
    .round(3)
)
print(summ.to_string())

print("\n=== By system, vs true eigenvalue error (the decisive test) ===")
tt = ev[ev.truth == "true eigenvalue error"]
print(
    tt.pivot_table(
        index="system", columns="criterion", values=["spearman", "top1_ok"], aggfunc="median"
    )
    .round(2)
    .to_string()
)

print("\n=== SC vs VAMP-2 agreement between themselves ===")
ag = [
    spearmanr(-g["sc"], g["vamp2"]).statistic
    for _, g in J.groupby(["system", "kind_n"])
    if g["sc"].nunique() > 2
]
print(f"median Spearman(-SC, VAMP2) across pools: {np.median(ag):.2f}")


joined: 5656 rows | with true_eig_err: 628 | with unified metrics: 1356

=== Does the SSL criterion select good kernels? (median over pools) ===
                                 median_spearman  top1_ok  median_rel_regret  n_pools
criterion truth                                                                      
sc        spectral bias (s-hat)           -0.550    0.172           3261.249       29
          spurious count                  -0.186    0.069              2.182       29
          true eigenvalue error            0.108    0.417              0.700       12
          unified score                   -0.285    0.000                NaN       29
vamp2     spectral bias (s-hat)           -0.715    0.069           2041.514       29
          spurious count                  -0.472    0.103              2.800       29
          true eigenvalue error           -0.299    0.417              0.302       12
          unified score                   -0.503    0.000                NaN     

In [11]:
# SSL encoders: does the SC criterion order encoders by quality?
ssl_sc = pd.read_csv(os.path.join(OUT, "ssl_sc_criteria.csv"))
ssl_scores = pd.read_csv(os.path.join(AN, "ssl", "ssl_scores.csv"))
ssl_scores["kind"] = ssl_scores["kind"].astype(str)
ssl_err = None
p = os.path.join(AN, "rescoring", "ssl_true_eig_err.csv")
if os.path.exists(p):
    ssl_err = pd.read_csv(p)

m = ssl_sc.merge(
    ssl_scores[
        [
            "system",
            "kind",
            "kernel",
            "method",
            "agg_bias_mean",
            "agg_dist_mean",
            "mean_spurious_residual_count",
        ]
    ],
    on=["system", "kind", "kernel"],
    how="left",
)
if ssl_err is not None:
    m = m.merge(ssl_err, on=["system", "kind", "kernel", "method"], how="left")

rows = []
for (sys_, kind, meth), g in m.groupby(["system", "kind", "method"]):
    for truth in ["true_eig_err", "agg_bias_mean", "mean_spurious_residual_count"]:
        if truth not in g.columns:
            continue
        q = sel_quality(g, "sc", truth, higher_better=False)
        if q:
            rows.append({"system": sys_, "kind": kind, "method": meth, "truth": truth} | q)
ssl_ev = pd.DataFrame(rows)
ssl_ev.to_csv(os.path.join(OUT, "ssl_criterion_quality.csv"), index=False)
if len(ssl_ev):
    print("SSL: SC (training val loss) as encoder-selection criterion:")
    print(ssl_ev.groupby("truth")[["spearman", "top1_ok"]].median().round(2).to_string())
else:
    print("SSL evaluation empty — check joins above.")


SSL: SC (training val loss) as encoder-selection criterion:
                              spearman  top1_ok
truth                                          
agg_bias_mean                     0.50      1.0
mean_spurious_residual_count      0.87      1.0


## Interpretation guide (fill the verdict from the printed tables)

- **The decisive row is criterion-vs-true-eigenvalue-error.** If VAMP-2/SC track true error on the
  reversible/normal systems but decouple on Duffing/logistic (where the operator is measurably non-normal,
  §4 of the analysis report), that is the thesis-shaped conclusion: *energy-based SSL criteria select
  singular-subspace quality, which coincides with eigen-fidelity exactly when the operator is normal* — the
  ICLR criterion inherits the same normal/non-normal boundary that organises RQ1–RQ4.
- SC and VAMP-2 should agree strongly with each other (both are singular-energy criteria); disagreement
  flags regularisation/whitening sensitivity, not substance.
- If the criterion beats the unified score against true error anywhere, that is a genuine finding in the
  criterion's favour (it is reference-free and cheap); if it loses everywhere on eigen-fidelity while
  winning on spurious counts, it complements rather than replaces the Kostic metrics.
- For encoders, SC-val ordering vs true error on linear/graph tests whether "pick the encoder with the best
  SSL loss" is sound — the practical default practitioners would use.

Outputs: `vamp_sc_criteria(.csv/_trials.csv)`, `ssl_sc_criteria.csv`, `criteria_joined.csv`,
`criterion_selection_quality.csv`, `ssl_criterion_quality.csv` in `analysis/vamp_selection/`.